# Texas RRC Form R-1 Extraction Pipeline

End-to-end pipeline for downloading, extracting, validating and comparing Form R-1 refinery filings.

**Three extraction tiers:**
- **Tier 1** — `pdfplumber` for digital PDFs (fast, ~100% accurate)
- **Tier 2** — Tesseract OCR with auto grid detection for clean scans (~60-75%)
- **Tier 3** — Claude API vision for difficult scans (~95%+, ~$0.01-0.03/page)

### Setup
```bash
pip install pdfplumber pytesseract Pillow numpy scipy pandas requests beautifulsoup4 openpyxl PyMuPDF anthropic
```

**Windows only:** Install Tesseract OCR from https://github.com/UB-Mannheim/tesseract/wiki  
Then uncomment the `pytesseract.pytesseract.tesseract_cmd` line in the setup cell below.

**For Tier 3 (optional):** Set your Anthropic API key:
```bash
set ANTHROPIC_API_KEY=sk-ant-...   # Windows
export ANTHROPIC_API_KEY=sk-ant-...  # Mac/Linux
```

---
## 0. Setup

In [11]:
import os, re, json, base64, io, tempfile, subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import pdfplumber
import pytesseract
from PIL import Image
from scipy.signal import find_peaks
import requests
from bs4 import BeautifulSoup

try:
    import fitz
    HAS_FITZ = True
except ImportError:
    HAS_FITZ = False
    print('WARNING: PyMuPDF not installed. Run: pip install PyMuPDF')

try:
    from anthropic import Anthropic
    HAS_ANTHROPIC = True
except ImportError:
    HAS_ANTHROPIC = False
    print('INFO: anthropic not installed — Tier 3 disabled. Run: pip install anthropic')

#try:
#    from paddleocr import PaddleOCR
    
    
#    os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'
    
#    paddle_ocr = PaddleOCR(use_angle_cls=False, lang='en')
#    HAS_PADDLE = True
#except ImportError:
#    HAS_PADDLE = False
#    paddle_ocr = None

try:
    from surya.foundation import FoundationPredictor
    from surya.recognition import RecognitionPredictor
    from surya.detection import DetectionPredictor
    surya_predictor = FoundationPredictor()
    surya_rec = RecognitionPredictor(surya_predictor)
    surya_det = DetectionPredictor()
    
    
    HAS_SURYA = True
except ImportError:
    HAS_SURYA = False
    surya_rec = surya_det = None


RAW_DATA_DIR = Path('Raw Data/Texas RRC')
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
EXISTING_DB_PATH = 'output_cleaned_v2_1.xlsx'

# Windows Tesseract — uncomment and adjust:
pytesseract.pytesseract.tesseract_cmd = r'C:\Users\luiscarlos.gaitan\AppData\Local\Programs\Tesseract-OCR\tesseract.exe'

PRODUCT_CODES = {1:'Propane',2:'Butane',3:'Butane-Propane',4:'Motor Gasoline',5:'Kerosene',6:'Home Heating Oil',7:'Diesel Fuel',8:'Other Middle Distillates',9:'Aviation Gasoline',10:'Kerosene-Type Jet Fuel',11:'Naphtha-Type Jet Fuel',12:'Fuel Oils #4 For Utility Use',13:'Fuel Oils #5,#6 For Utility Use',14:'Fuel Oils #4 For Non-Utility Use',15:'Fuel Oils #5,#6 For Non-Utility Use',16:'Bunker C',17:'Navy Special',18:'Other Residual Fuels',19:'Petrochemical Feedstock',20:'Lubricants',21:'Special Naphthas',22:'Solvent Products',23:'Miscellaneous',24:'Crude Oil'}
DATA_COLUMNS = ['Storage Beginning of Month','Receipts','Input','Products Manufactured','Fuel Used','Deliveries','Storage End of Month']

print(f'Setup complete. PyMuPDF: {HAS_FITZ}, PaddleOCR: {HAS_PADDLE}, Claude: {HAS_ANTHROPIC}')

AttributeError: 'SuryaDecoderConfig' object has no attribute 'pad_token_id'

In [12]:
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

foundation_predictor = FoundationPredictor()
surya_rec = RecognitionPredictor(foundation_predictor)
surya_det = DetectionPredictor()
HAS_SURYA = True

AttributeError: 'SuryaDecoderConfig' object has no attribute 'pad_token_id'

---
## 1. Core Extraction Engine

In [ ]:
# ── Shared helpers ──

def parse_number(raw) -> float:
    if raw is None: return 0.0
    s = str(raw).strip().replace(',','').replace(' ','').replace('|','')
    s = re.sub(r'[^0-9.\-]', '', s)
    if not s or s == '-': return 0.0
    try: return float(s)
    except ValueError: return 0.0

def rasterize_page(pdf_path, dpi=200):
    if HAS_FITZ:
        doc = fitz.open(pdf_path)
        zoom = dpi / 72
        pix = doc[0].get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
        doc.close()
        return img
    with tempfile.TemporaryDirectory() as d:
        subprocess.run(['pdftoppm','-jpeg','-r',str(dpi),'-f','1','-l','1',pdf_path,f'{d}/p'], check=True, capture_output=True)
        return Image.open(sorted(Path(d).glob('p-*.jpg'))[0]).copy()

def is_digital_pdf(path):
    try:
        with pdfplumber.open(path) as pdf:
            return len((pdf.pages[0].extract_text() or '').strip()) > 200
    except: return False

def extract_header_text(text):
    h = {}
    m = re.search(r'Operator[:\s]+(.+?)(?:Month|Address|$)', text, re.IGNORECASE)
    if m: h['operator'] = m.group(1).strip().rstrip('.')
    m = re.search(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}', text, re.IGNORECASE)
    if m: h['month_str'] = m.group(0)
    m = re.search(r'(\d{2}-\d{4})', text)
    if m: h['registration_no'] = m.group(1)
    m = re.search(r'Refinery\s*(?:Name)?\s*[:.]?\s*(.+?)(?:\n|$)', text, re.IGNORECASE)
    if m: h['refinery_name'] = m.group(1).strip()
    return h

def validate_balance(df):
    df = df.copy()
    df['balance_diff'] = (df['Storage Beginning of Month'] + df['Receipts'] + df['Products Manufactured']) - (df['Input'] + df['Fuel Used'] + df['Deliveries'] + df['Storage End of Month'])
    df['balanced'] = df['balance_diff'].abs() < 2
    return df

def balance_stats(df):
    active = df[df[DATA_COLUMNS].sum(axis=1) > 0]
    n = len(active)
    b = int(active['balanced'].sum()) if n else 0
    return {'total_active': n, 'balanced_count': b, 'pass_rate': round(b/n, 3) if n else 1.0}

print('Helpers loaded.')

In [ ]:
# ── Tier 1: Digital PDF ──

def extract_digital(path):
    with pdfplumber.open(path) as pdf:
        text = pdf.pages[0].extract_text() or ''
        header = extract_header_text(text)
        tables = pdf.pages[0].extract_tables()
        if not tables:
            return {'header':header,'data':[],'mode':'digital','tier':3,'error':'No table found','balance_stats':{}}
        rows = []
        for r in tables[0][1:]:
            if r[0] and 'TOTALS' in str(r[0]): continue
            c = parse_number(r[1])
            if c == 0: continue
            rows.append({'Code':int(c), 'Storage Beginning of Month':parse_number(r[2]),
                'Receipts':parse_number(r[3]), 'Input':parse_number(r[4]),
                'Products Manufactured':parse_number(r[5]), 'Fuel Used':parse_number(r[6]),
                'Deliveries':parse_number(r[7]), 'Storage End of Month':parse_number(r[8])})
        df = validate_balance(pd.DataFrame(rows)) if rows else pd.DataFrame()
        return {'header':header,'data':rows,'mode':'digital','tier':1,'error':None,
                'balance_stats':balance_stats(df) if not df.empty else {}}

print('Tier 1 loaded.')

In [ ]:
# ── Tier 2: Tesseract OCR with auto grid detection ──

def detect_grid(img):
    arr = np.array(img.convert('L'))
    th = 100
    rk = (arr < th).sum(axis=1)
    hp, _ = find_peaks(rk, height=arr.shape[1]*0.15, distance=12)
    hp = hp[(hp >= arr.shape[0]*0.15) & (hp <= arr.shape[0]*0.9)]
    if len(hp) < 10: return None, None
    sp = np.diff(hp)
    di = np.where((sp >= 18) & (sp <= 50))[0]
    if len(di) < 5: return None, None
    th_list = hp[max(0,di[0]-2):min(len(hp),di[-1]+3)].tolist()
    tr = arr[th_list[0]:th_list[-1], :]
    ck = (tr < th).sum(axis=0)
    vp, _ = find_peaks(ck, height=len(tr)*0.25, distance=25)
    return vp.tolist(), th_list

def id_columns(vl):
    if len(vl) < 4: return []
    gaps = [(vl[i], vl[i+1], vl[i+1]-vl[i]) for i in range(len(vl)-1)]
    wide = [(a,b,w) for a,b,w in gaps if w > 100]
    if len(wide) >= 8: return [(a,b) for a,b,w in wide[1:8]]
    if len(wide) == 7: return [(a,b) for a,b,w in wide]
    return [(a,b) for a,b,w in gaps[-7:]] if len(gaps) >= 7 else []

def find_start(hl, vl, img):
    if not vl or len(vl) < 4: return min(2, len(hl)-25)
    gaps = [(vl[i],vl[i+1],vl[i+1]-vl[i],i) for i in range(len(vl)-1)]
    lh = [g for g in gaps if g[0] < img.width*0.4]
    if not lh: return min(2, len(hl)-25)
    mi = max(lh, key=lambda g:g[2])[3]
    cx1, cx2 = vl[mi+1], vl[min(mi+2,len(vl)-1)]
    for i in range(min(15,len(hl)-1)):
        y1,y2 = hl[i]+2, hl[i+1]-2
        if y2-y1 < 8: continue
        c = img.crop((cx1,y1,cx2,y2))
        w,h = c.size
        if w<5 or h<5: continue
        c = c.resize((w*4,h*4), Image.LANCZOS)
        t = pytesseract.image_to_string(c, config='--psm 7 -c tessedit_char_whitelist=0123456789').strip()
        if t == '1': return i
    sp = [hl[i+1]-hl[i] for i in range(len(hl)-1)]
    for i in range(1,len(sp)):
        if sp[i] < 45 and sp[i-1] >= 45: return i
    return min(2, max(0, len(hl)-26))

def extract_scanned(path, dpi=200):
    try: img = rasterize_page(path, dpi)
    except Exception as e:
        return {'header':{},'data':[],'mode':'scanned_ocr','tier':3,'error':str(e),'balance_stats':{}}
    header = extract_header_text(pytesseract.image_to_string(img.crop((0,0,img.width,int(img.height*0.25)))))
    vl, hl = detect_grid(img)
    if not vl or not hl:
        return {'header':header,'data':[],'mode':'scanned_ocr','tier':3,'error':'No grid','balance_stats':{}}
    cb = id_columns(vl)
    if len(cb) < 7:
        return {'header':header,'data':[],'mode':'scanned_ocr','tier':3,'error':f'{len(cb)} cols','balance_stats':{}}
    ds = find_start(hl, vl, img)
    PAD, SCALE = 3, 4
    rows = []
    for code in range(1,25):
        hi = ds + code - 1
        if hi+1 >= len(hl): break
        yt, yb = hl[hi]+PAD, hl[hi+1]-PAD
        if yb <= yt: continue
        rd = {'Code': code}
        for ci, cn in enumerate(DATA_COLUMNS):
            if ci >= len(cb): rd[cn] = 0.0; continue
            x1,x2 = cb[ci]
            cell = img.crop((x1+PAD, yt, x2-PAD, yb))
            w,h = cell.size
            if w<5 or h<5: rd[cn] = 0.0; continue
            cell = cell.resize((w*SCALE, h*SCALE), Image.LANCZOS)
            t = pytesseract.image_to_string(cell, config='--psm 7 -c tessedit_char_whitelist=0123456789,.-').strip()
            rd[cn] = parse_number(t)
        rows.append(rd)
    existing = {r['Code'] for r in rows}
    for c in range(1,25):
        if c not in existing: rows.append({**{col:0.0 for col in DATA_COLUMNS}, 'Code':c})
    rows.sort(key=lambda r: r['Code'])
    df = validate_balance(pd.DataFrame(rows))
    st = balance_stats(df)
    tier = 1 if st['pass_rate']>=0.75 else 2 if st['pass_rate']>=0.25 else 3
    return {'header':header,'data':rows,'mode':'scanned_ocr','tier':tier,'error':None,'balance_stats':st}

print('Tier 2 loaded.')

In [ ]:
# ── Tier 2B: PaddleOCR ──

def paddle_ocr_cell(cell_img):
    img_arr = np.array(cell_img)
    result = paddle_ocr.ocr(img_arr, cls=False)
    if not result or not result[0]: return 0.0, 0.0
    texts, min_conf = [], 1.0
    for line in result[0]:
        text, conf = line[1]
        texts.append(text); min_conf = min(min_conf, conf)
    return parse_number(' '.join(texts)), min_conf

def extract_scanned_paddle(path, dpi=200):
    if not HAS_PADDLE:
        return {'header':{},'data':[],'mode':'paddle_ocr','tier':3,'error':'not installed','balance_stats':{}}
    try: img = rasterize_page(path, dpi)
    except Exception as e:
        return {'header':{},'data':[],'mode':'paddle_ocr','tier':3,'error':str(e),'balance_stats':{}}
    header = extract_header_text(pytesseract.image_to_string(img.crop((0,0,img.width,int(img.height*0.25)))))
    vl, hl = detect_grid(img)
    if not vl or not hl:
        return {'header':header,'data':[],'mode':'paddle_ocr','tier':3,'error':'No grid','balance_stats':{}}
    cb = id_columns(vl)
    if len(cb) < 7:
        return {'header':header,'data':[],'mode':'paddle_ocr','tier':3,'error':f'{len(cb)} cols','balance_stats':{}}
    ds = find_start(hl, vl, img)
    PAD, SCALE = 3, 3
    rows, total_conf, n_cells = [], 0, 0
    for code in range(1,25):
        hi = ds + code - 1
        if hi+1 >= len(hl): break
        yt, yb = hl[hi]+PAD, hl[hi+1]-PAD
        if yb <= yt: continue
        rd = {'Code': code}
        for ci, cn in enumerate(DATA_COLUMNS):
            if ci >= len(cb): rd[cn] = 0.0; continue
            x1, x2 = cb[ci]
            cell = img.crop((x1+PAD, yt, x2-PAD, yb))
            w, h = cell.size
            if w<5 or h<5: rd[cn] = 0.0; continue
            cell = cell.resize((w*SCALE, h*SCALE), Image.LANCZOS)
            val, conf = paddle_ocr_cell(cell)
            rd[cn] = val; total_conf += conf; n_cells += 1
        rows.append(rd)
    existing = {r['Code'] for r in rows}
    for c in range(1,25):
        if c not in existing: rows.append({**{col:0.0 for col in DATA_COLUMNS},'Code':c})
    rows.sort(key=lambda r: r['Code'])
    df = validate_balance(pd.DataFrame(rows)); st = balance_stats(df)
    tier = 1 if st['pass_rate']>=0.75 else 2 if st['pass_rate']>=0.25 else 3
    return {'header':header,'data':rows,'mode':'paddle_ocr','tier':tier,'error':None,
            'balance_stats':st,'avg_confidence':round(total_conf/n_cells,3) if n_cells else 0}

print('Tier 2B (PaddleOCR) loaded.')


In [ ]:
# ── Tier 3: Claude API vision ──

CLAUDE_PROMPT = '''You are extracting data from a Texas Railroad Commission Form R-1 (Monthly Report and Operations Statement for Refineries).

Look at this scanned page and extract the material balance table.
Return ONLY a JSON object — no markdown, no backticks, no explanation:

{"operator":"...","month":"...","registration_no":"XX-XXXX","refinery_name":"...","rows":[{"code":1,"storage_bom":0,"receipts":0,"input":0,"products_mfg":0,"fuel_used":0,"deliveries":0,"storage_eom":0}]}

Rules: Include ALL 24 codes (1-24). Use 0 for blank/dash cells. Integers only, no commas. Return ONLY valid JSON.'''

def extract_claude(path, model='claude-sonnet-4-5-20250514'):
    if not HAS_ANTHROPIC:
        return {'header':{},'data':[],'mode':'claude_vision','tier':3,'error':'anthropic not installed','balance_stats':{}}
    try: img = rasterize_page(path, 200)
    except Exception as e:
        return {'header':{},'data':[],'mode':'claude_vision','tier':3,'error':str(e),'balance_stats':{}}
    buf = io.BytesIO(); img.save(buf, format='JPEG', quality=90)
    b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    try:
        msg = Anthropic().messages.create(model=model, max_tokens=4096, messages=[{
            'role':'user','content':[
                {'type':'image','source':{'type':'base64','media_type':'image/jpeg','data':b64}},
                {'type':'text','text':CLAUDE_PROMPT}]}])
    except Exception as e:
        return {'header':{},'data':[],'mode':'claude_vision','tier':3,'error':str(e),'balance_stats':{}}
    txt = re.sub(r'^```(?:json)?\s*','', msg.content[0].text.strip())
    txt = re.sub(r'\s*```$','', txt)
    try: p = json.loads(txt)
    except json.JSONDecodeError as e:
        return {'header':{},'data':[],'mode':'claude_vision','tier':3,'error':f'JSON: {e}','balance_stats':{}}
    header = {'operator':p.get('operator',''),'month_str':p.get('month',''),
              'registration_no':p.get('registration_no',''),'refinery_name':p.get('refinery_name','')}
    rows = []
    for r in p.get('rows',[]):
        c = int(r.get('code',0))
        if c<1 or c>24: continue
        rows.append({'Code':c,'Storage Beginning of Month':float(r.get('storage_bom',0)),
            'Receipts':float(r.get('receipts',0)),'Input':float(r.get('input',0)),
            'Products Manufactured':float(r.get('products_mfg',0)),'Fuel Used':float(r.get('fuel_used',0)),
            'Deliveries':float(r.get('deliveries',0)),'Storage End of Month':float(r.get('storage_eom',0))})
    existing = {r['Code'] for r in rows}
    for c in range(1,25):
        if c not in existing: rows.append({**{col:0.0 for col in DATA_COLUMNS},'Code':c})
    rows.sort(key=lambda r:r['Code'])
    df = validate_balance(pd.DataFrame(rows)); st = balance_stats(df)
    cost = (msg.usage.input_tokens*3 + msg.usage.output_tokens*15)/1_000_000
    return {'header':header,'data':rows,'mode':'claude_vision',
            'tier':1 if st['pass_rate']>=0.75 else 2,'error':None,'balance_stats':st,
            'api_usage':{'input_tokens':msg.usage.input_tokens,'output_tokens':msg.usage.output_tokens,'cost_usd':round(cost,4)}}

print('Tier 3 loaded.')

In [ ]:
# ── Unified extractor with OCR engine selection ──

def extract_r1(path, use_claude=True, claude_threshold=0.25, ocr_engine='auto'):
    """
    Extract Form R-1 from any PDF.
    ocr_engine: 'tesseract', 'paddle', or 'auto' (tries both, picks best)
    """
    if is_digital_pdf(path):
        r = extract_digital(path)
        if r['data']: return r
    
    if ocr_engine == 'auto' and HAS_PADDLE:
        r_tess = extract_scanned(path)
        r_pad = extract_scanned_paddle(path)
        tr = r_tess.get('balance_stats',{}).get('pass_rate',0)
        pr = r_pad.get('balance_stats',{}).get('pass_rate',0)
        r = r_pad if pr > tr else r_tess
        r['ocr_comparison'] = {'tesseract':tr,'paddle':pr,'winner':r['mode']}
    elif ocr_engine == 'paddle' and HAS_PADDLE:
        r = extract_scanned_paddle(path)
    else:
        r = extract_scanned(path)
    
    if r.get('data') and r.get('balance_stats',{}).get('pass_rate',0) >= claude_threshold:
        return r
    t2 = r
    if use_claude and HAS_ANTHROPIC:
        cr = extract_claude(path)
        if not cr['error']: return cr
    if t2.get('data'):
        t2['note'] = 'Low confidence; Claude unavailable or failed'
        return t2
    return {'header':{},'data':[],'mode':'failed','tier':3,'error':'All failed','balance_stats':{}}

print('Unified extractor ready (Tesseract + PaddleOCR + Claude).')


---
## 2. Download PDFs

In [ ]:
def download_pdfs(year_start, year_end, month_start=1, month_end=12, facility_filter=None):
    log = []
    for year in range(year_start, year_end+1):
        ms = month_start if year == year_start else 1
        me = month_end if year == year_end else 12
        for month in range(ms, me+1):
            if year < 2020:
                url = f'https://www.rrc.texas.gov/oil-and-gas/research-and-statistics/refinery-statements/refineries-statements-{year}/refinery-statements-{month:02d}-{year}'
            else:
                url = f'https://www.rrc.texas.gov/oil-and-gas/research-and-statistics/refinery-statements/refineries-statements-{year}/refinery-statements-{month}-{year}'
            print(f'{year}-{month:02d}...', end=' ')
            try:
                soup = BeautifulSoup(requests.get(url, timeout=30).text, 'html.parser')
                table = soup.find('table', {'class': 'table table-bordered'})
                if not table: print('no table'); continue
            except Exception as e: print(f'FAIL: {e}'); continue
            links = []
            for tr in table.find_all('tr')[1:]:
                tds = tr.find_all('td')
                if len(tds) < 2: continue
                fid, op = tds[0].text.strip(), tds[1].text.strip()
                a = None
                for td in tds:
                    tag = td.find('a')
                    if tag and tag.get('href'):
                        a = tag['href']
                        if not a.startswith('http'): a = 'https://www.rrc.texas.gov' + a
                        break
                if a and fid: links.append((fid, op, a))
            if facility_filter: links = [(f,o,l) for f,o,l in links if f in facility_filter]
            print(f'{len(links)} PDFs')
            for fid, op, link in links:
                d = RAW_DATA_DIR / fid; d.mkdir(parents=True, exist_ok=True)
                fn = f'{year}_{month}___{fid}_{re.sub(r"[<>:\"/\\|?*]", "_", op)}.pdf'
                fp = d / fn
                if fp.exists(): log.append({'file':str(fp),'status':'exists'}); continue
                try:
                    fp.write_bytes(requests.get(link, timeout=60).content)
                    log.append({'file':str(fp),'status':'ok'})
                except Exception as e: log.append({'file':str(fp),'status':'fail','error':str(e)})
    return pd.DataFrame(log)

In [ ]:
# Uncomment to download:
# dl = download_pdfs(2024, 2025)
# dl = download_pdfs(2025, 2025, facility_filter=['03-0052','03-0138','02-0057'])
print('Download cell ready.')

---
## 3. Batch Extraction

In [ ]:
def batch_extract(data_dir=RAW_DATA_DIR, facility_filter=None, use_claude=True, ocr_engine='auto'):
    pdfs = sorted(data_dir.rglob('*.pdf'))
    if facility_filter: pdfs = [p for p in pdfs if any(f in str(p) for f in facility_filter)]
    print(f'Processing {len(pdfs)} PDFs...')
    all_rows, log = [], []
    for i, fp in enumerate(pdfs):
        if (i+1) % 25 == 0: print(f'  [{i+1}/{len(pdfs)}]')
        parts = fp.stem.split('___')
        if len(parts) < 2: log.append({'file':str(fp),'error':'bad name'}); continue
        fid = str(parts[1].split('_')[0])
        try:
            yr, mo = parts[0].split('_')[:2]; yr, mo = int(yr), int(mo)
            date = datetime(yr, mo+1, 1) - pd.Timedelta(days=1) if mo < 12 else datetime(yr,12,31)
        except: date = None
        r = extract_r1(str(fp), use_claude=use_claude, ocr_engine=ocr_engine)
        log.append({'file':str(fp),'id':fid,'date':date,'mode':r['mode'],'tier':r.get('tier',3),
                    'error':r.get('error'),'active':r.get('balance_stats',{}).get('total_active',0),
                    'balanced':r.get('balance_stats',{}).get('balanced_count',0),
                    'pass_rate':r.get('balance_stats',{}).get('pass_rate',0),
                    'cost':r.get('api_usage',{}).get('cost_usd',0)})
        for row in r.get('data',[]):
            row.update({'id':fid,'date':date,'extraction_mode':r['mode'],'extraction_tier':r.get('tier',3)})
            all_rows.append(row)
    rdf, ldf = pd.DataFrame(all_rows), pd.DataFrame(log)
    print(f'\nDone: {len(rdf)} rows, {len(ldf[ldf["error"].notna()])} errors')
    if not ldf.empty:
        print(f'Tiers: {dict(ldf["tier"].value_counts().sort_index())}')
        print(f'Modes: {dict(ldf["mode"].value_counts())}')
        c = ldf['cost'].sum()
        if c > 0: print(f'API cost: ${c:.2f}')
    return rdf, ldf

In [ ]:
# Uncomment to extract:
# new_data, new_log = batch_extract()
# new_data, new_log = batch_extract(facility_filter=['03-0052','03-0138','02-0057'])
# new_data, new_log = batch_extract(use_claude=False)  # Tesseract only

# Save:
# new_data.to_csv(OUTPUT_DIR / 'extracted_data.csv', index=False)
# new_log.to_csv(OUTPUT_DIR / 'extraction_log.csv', index=False)
print('Batch cell ready.')

---
## 4. Compare to Existing Database

In [ ]:
def compare(new_df, existing_path=EXISTING_DB_PATH):
    ex = pd.read_excel(existing_path, dtype={'id':str})
    ex['date'] = pd.to_datetime(ex['date'])
    nw = new_df.copy(); nw['date'] = pd.to_datetime(nw['date'])
    m = nw.merge(ex, on=['id','date','Code'], how='outer', suffixes=('_new','_old'), indicator=True)
    new_only = m[m['_merge']=='left_only']
    old_only = m[m['_merge']=='right_only']
    both = m[m['_merge']=='both']
    oc = [f'{c}_old' for c in DATA_COLUMNS if f'{c}_old' in both.columns]
    nc = [f'{c}_new' for c in DATA_COLUMNS if f'{c}_new' in both.columns]
    silent = both[(both[oc].fillna(0).sum(axis=1)==0) & (both[nc].fillna(0).sum(axis=1)>0)]
    print(f'Both: {len(both)} | Silent zeros recovered: {len(silent)} | New only: {len(new_only)} | Existing only: {len(old_only)}')
    if not silent.empty:
        for (fid,dt), g in silent.groupby(['id','date']):
            print(f'  {fid} {dt.strftime("%Y-%m")}: {len(g)} codes recovered')
    return {'silent_zeros':silent,'new_only':new_only,'existing_only':old_only}

In [ ]:
# Uncomment after extraction:
# comp = compare(new_data)
print('Compare cell ready.')

---
## 5a. A/B Test: Tesseract vs PaddleOCR

Compare both OCR engines on scanned PDFs to find which works better for your data.

In [ ]:
def ab_test(pdf_paths):
    results = []
    for p in pdf_paths:
        if not os.path.exists(p): print(f'  Skip: {p}'); continue
        name = os.path.basename(p)
        print(f'  {name}...', end=' ')
        rt = extract_scanned(str(p))
        ts = rt.get('balance_stats',{})
        if HAS_PADDLE:
            rp = extract_scanned_paddle(str(p))
            ps = rp.get('balance_stats',{})
        else:
            ps = {'total_active':0,'balanced_count':0,'pass_rate':0}; rp = {}
        tr, pr = ts.get('pass_rate',0), ps.get('pass_rate',0)
        w = 'paddle' if pr>tr else 'tesseract' if tr>pr else 'tie'
        print(f'tess={tr:.0%} paddle={pr:.0%} -> {w}')
        results.append({'file':name,'tess_rate':tr,'paddle_rate':pr,
            'paddle_conf':rp.get('avg_confidence',0),'winner':w})
    df = pd.DataFrame(results)
    if not df.empty:
        print(f'\nTesseract avg: {df["tess_rate"].mean():.1%}')
        print(f'PaddleOCR avg: {df["paddle_rate"].mean():.1%}')
        print(f'Winners: {dict(df["winner"].value_counts())}')
    return df

# test_files = [
#     '2026-january-03-0138.pdf',
#     '2025_1___02-0057_DIAMOND SHAMROCK REFINING COMPANY.pdf',
# ]
# ab_results = ab_test(test_files)
print('A/B test cell ready.')


---
## 5. Quick Test: Single PDFs

In [ ]:
def test_pdf(path, use_claude=True):
    if not os.path.exists(path): print(f'Not found: {path}'); return None
    r = extract_r1(path, use_claude=use_claude)
    print(f'File: {os.path.basename(path)}')
    print(f'Mode: {r["mode"]} | Tier: {r.get("tier","?")} | Header: {r["header"]}')
    if r.get('error'): print(f'Error: {r["error"]}')
    s = r.get('balance_stats',{})
    if s: print(f'Balance: {s.get("balanced_count",0)}/{s.get("total_active",0)} pass ({s.get("pass_rate",0):.0%})')
    if r.get('api_usage'): print(f'API cost: ${r["api_usage"]["cost_usd"]:.4f}')
    if not r['data']: return None
    df = validate_balance(pd.DataFrame(r['data']))
    df['material'] = df['Code'].map(PRODUCT_CODES)
    df['confidence'] = df['balanced'].map({True:'ok', False:'REVIEW'})
    pd.set_option('display.float_format', '{:,.0f}'.format)
    print(df[['Code','material']+DATA_COLUMNS+['balance_diff','confidence']].to_string(index=False))
    return df

In [ ]:
# ── Test digital PDF (Blanchard) ──
# df = test_pdf('2025-june-03-0052.pdf')
print('Uncomment and update path.')

In [ ]:
# ── Test clean scan (ExxonMobil) ──
# df = test_pdf('2026-january-03-0138.pdf')
print('Uncomment and update path.')

In [ ]:
# ── Test difficult scan (Diamond Shamrock) — will escalate to Tier 3 ──
# df = test_pdf('2025_1___02-0057_DIAMOND SHAMROCK REFINING COMPANY.pdf')

# Without Claude:
# df = test_pdf('2025_1___02-0057_DIAMOND SHAMROCK REFINING COMPANY.pdf', use_claude=False)
print('Uncomment and update path.')

---
## 6. Merge & Save Final Database

In [ ]:
def merge_db(new_df, existing_path=EXISTING_DB_PATH):
    ex = pd.read_excel(existing_path, dtype={'id':str})
    ex['date'] = pd.to_datetime(ex['date'])
    nw = new_df[['id','date','Code']+DATA_COLUMNS].copy()
    nw['date'] = pd.to_datetime(nw['date'])
    mg = ex.merge(nw, on=['id','date','Code'], how='outer', suffixes=('','_new'))
    for col in DATA_COLUMNS:
        nc = f'{col}_new'
        if nc in mg.columns:
            z = mg[col].fillna(0)==0; h = mg[nc].fillna(0)!=0
            mg.loc[z&h, col] = mg.loc[z&h, nc]
            mg[col] = mg[col].fillna(mg[nc])
            mg.drop(columns=[nc], inplace=True)
    mg = mg.sort_values(['id','date','Code']).reset_index(drop=True)
    print(f'Merged: {len(mg)} rows, {mg["id"].nunique()} facilities')
    return mg

In [ ]:
# Uncomment after extraction:
# final = merge_db(new_data)
# final.to_csv(OUTPUT_DIR / 'rrc_r1_database.csv', index=False)
# final.to_excel(OUTPUT_DIR / 'rrc_r1_database.xlsx', index=False)
print('Merge cell ready.')

---
## Notes

| Tier | Method | Accuracy | Cost |
|------|--------|----------|------|
| 1 | pdfplumber | ~100% | Free |
| 2a | Tesseract OCR | 60-75% | Free |
| 2b | PaddleOCR | Run A/B test | Free |
| 3 | Claude vision | ~95%+ | ~$0.01-0.03/pg |

**OCR engine:** `ocr_engine='auto'` runs both Tesseract and PaddleOCR, picks the winner by balance pass rate.  
Use `'tesseract'` or `'paddle'` to force one. Run A/B test (5a) to see which is better for your PDFs.

**Install PaddleOCR:** `pip install paddlepaddle paddleocr` (CPU) or `pip install paddlepaddle-gpu paddleocr` (GPU)

**ID dtype:** All IDs forced to `str`. When reading CSVs: `pd.read_csv('f.csv', dtype={'id':str})`

**Balance check:** ±1 = rounding, ±1K = measurement, ±100K+ from scan = likely OCR error.
